# Top-1 vs Top-5 probability analysis

This notebook analyzes the step-126,000 checkpoint on the full development set. It separates examples into:

1. **Top-1 correct**
2. **Top-1 incorrect, recovered by top-5**
3. **Outside top-5**

Probabilities are normalized over all root-BPE candidates allowed by each first-character hint, matching the model's restricted prediction objective. They are not probabilities over the complete GPT-2 vocabulary. The expensive inference result is cached as a CSV so plotting cells can be rerun quickly.


In [ ]:
from __future__ import annotations

import gc
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file()
)
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from contest1 import hint_masked_lib as hml
from contest1.train_hint_masked import validate_dev_frame

CHECKPOINT = PROJECT_ROOT / 'artifacts/gpt2-keyboard-script/best_dev_step_126000.pt'
TRAIN_PATH = PROJECT_ROOT / 'train/train.src.tok'
DEV_PATH = PROJECT_ROOT / 'data/devv_eval.csv'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts/gpt2-keyboard-script/analysis_step_126000/probabilities'
CACHE_PATH = OUTPUT_DIR / 'full_dev_candidate_probabilities_batch64.csv'
MODEL_NAME = 'gpt2'
TOP_K = 5
BATCH_SIZE = 64
MAX_CONTEXT_TOKENS = 256
RECOMPUTE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
def position_ids(attention_mask: torch.Tensor) -> torch.Tensor:
    positions = attention_mask.long().cumsum(dim=1) - 1
    return positions.masked_fill(attention_mask == 0, 0)


@torch.inference_mode()
def calculate_candidate_probabilities() -> pd.DataFrame:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    tokenizer.truncation_side = 'left'

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    model.config.pad_token_id = tokenizer.pad_token_id
    torch.nn.Module.to(model, device)
    model.eval()

    lexicon = hml.build_training_lexicon(tokenizer, TRAIN_PATH)
    dev = pd.read_csv(DEV_PATH, keep_default_na=False)
    validate_dev_frame(dev)
    candidate_tensors = {
        hint: torch.tensor(ids, dtype=torch.long, device=device)
        for hint, ids in lexicon.candidate_ids_by_hint.items()
    }
    candidate_positions = {
        hint: {token_id: index for index, token_id in enumerate(ids)}
        for hint, ids in lexicon.candidate_ids_by_hint.items()
    }
    representative_roots = {
        hint: {word: root for root, word in mapping.items()}
        for hint, mapping in lexicon.hint_token_word.items()
    }

    records = []
    for start in tqdm(range(0, len(dev), BATCH_SIZE), desc='Candidate probabilities'):
        batch = dev.iloc[start:start + BATCH_SIZE]
        encoded = tokenizer(
            batch['context'].tolist(),
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_CONTEXT_TOKENS,
        ).to(device)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **encoded,
                position_ids=position_ids(encoded['attention_mask']),
                logits_to_keep=1,
            ).logits[:, -1, :]

        for row_index, (_, example) in enumerate(batch.iterrows()):
            hint = example['first letter']
            answer = example['answer']
            candidate_ids = candidate_tensors.get(hint)
            if candidate_ids is None or candidate_ids.numel() == 0:
                prediction = lexicon.fallback_by_hint.get(hint, hint)
                words = [prediction]
                probabilities = [1.0]
                candidate_count = 1
                entropy = 0.0
                answer_probability = 1.0 if answer == prediction else np.nan
            else:
                restricted_logits = logits[row_index].index_select(0, candidate_ids).float()
                all_probabilities = restricted_logits.softmax(dim=0)
                count = min(TOP_K, candidate_ids.numel())
                probabilities_tensor, indices = torch.topk(all_probabilities, k=count)
                roots = candidate_ids.index_select(0, indices).tolist()
                words = [lexicon.hint_token_word[hint][int(root)] for root in roots]
                probabilities = probabilities_tensor.tolist()
                prediction = words[0]
                candidate_count = int(candidate_ids.numel())
                entropy_value = -(all_probabilities * all_probabilities.clamp_min(1e-12).log()).sum()
                entropy = float(entropy_value / np.log(candidate_count)) if candidate_count > 1 else 0.0
                answer_root = representative_roots.get(hint, {}).get(answer)
                answer_position = candidate_positions.get(hint, {}).get(answer_root)
                answer_probability = (
                    float(all_probabilities[answer_position])
                    if answer_position is not None
                    else np.nan
                )

            top_1_correct = prediction == answer
            top_5_correct = answer in words
            answer_rank = words.index(answer) + 1 if top_5_correct else np.nan
            top_2_probability = probabilities[1] if len(probabilities) > 1 else 0.0
            if top_1_correct:
                outcome = 'Top-1 correct'
            elif top_5_correct:
                outcome = 'Top-5 recovery'
            else:
                outcome = 'Outside top-5'
            records.append({
                'row': int(start + row_index),
                'first_letter': hint,
                'answer': answer,
                'top_1_prediction': prediction,
                'top_5_predictions': json.dumps(words, ensure_ascii=False),
                'top_5_probabilities': json.dumps(probabilities),
                'top_1_correct': top_1_correct,
                'top_5_correct': top_5_correct,
                'outcome': outcome,
                'answer_rank': answer_rank,
                'answer_probability': answer_probability,
                'top_1_probability': probabilities[0],
                'top_2_probability': top_2_probability,
                'top_1_margin': probabilities[0] - top_2_probability,
                'top_5_probability_mass': sum(probabilities),
                'normalized_entropy': entropy,
                'candidate_count': candidate_count,
            })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(records)


if RECOMPUTE or not CACHE_PATH.is_file():
    probabilities = calculate_candidate_probabilities()
    probabilities.to_csv(CACHE_PATH, index=False)
else:
    probabilities = pd.read_csv(CACHE_PATH)

print(f'Loaded {len(probabilities):,} examples from {CACHE_PATH}')


## Outcome and confidence summary

The three groups distinguish correct top-1 predictions, errors rescued by ranks 2–5, and answers absent from the top five. Comparing top-1 confidence and margin reveals whether errors are uncertain or confidently wrong.


In [ ]:
outcome_order = ['Top-1 correct', 'Top-5 recovery', 'Outside top-5']
probabilities['outcome'] = pd.Categorical(
    probabilities['outcome'], categories=outcome_order, ordered=True
)

top_1_accuracy = probabilities['top_1_correct'].mean()
top_5_accuracy = probabilities['top_5_correct'].mean()
print(f'Top-1 accuracy: {top_1_accuracy:.3%}')
print(f'Top-5 accuracy: {top_5_accuracy:.3%}')
print(f'Errors recovered by top-5: {(top_5_accuracy - top_1_accuracy):.3%}')

summary = probabilities.groupby('outcome', observed=False).agg(
    examples=('row', 'size'),
    mean_top_1_probability=('top_1_probability', 'mean'),
    median_top_1_probability=('top_1_probability', 'median'),
    mean_top_1_margin=('top_1_margin', 'mean'),
    mean_top_5_mass=('top_5_probability_mass', 'mean'),
    mean_answer_probability=('answer_probability', 'mean'),
    mean_normalized_entropy=('normalized_entropy', 'mean'),
).reset_index()
summary['share'] = summary['examples'] / len(probabilities)
summary


In [ ]:
colors = {
    'Top-1 correct': '#2a9d8f',
    'Top-5 recovery': '#e9c46a',
    'Outside top-5': '#e76f51',
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_specs = [
    ('top_1_probability', 'Top-1 probability'),
    ('top_1_margin', 'Top-1 minus top-2 probability'),
    ('top_5_probability_mass', 'Cumulative top-5 probability'),
    ('normalized_entropy', 'Normalized candidate entropy'),
]
bins = np.linspace(0, 1, 41)
for axis, (column, title) in zip(axes.flat, plot_specs):
    for outcome in outcome_order:
        values = probabilities.loc[probabilities['outcome'] == outcome, column].dropna()
        axis.hist(
            values, bins=bins, density=True, histtype='step', linewidth=2,
            color=colors[outcome], label=f'{outcome} (n={len(values):,})',
        )
    axis.set(title=title, xlabel='Probability / normalized value', ylabel='Density', xlim=(0, 1))
    axis.grid(alpha=0.2)
    axis.legend()
fig.suptitle('Candidate probability distributions by prediction outcome', fontsize=16)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'probability_distributions.png', dpi=160, bbox_inches='tight')
plt.show()


## What top-5 recovers

For top-1 errors, this section compares the probability assigned to the correct answer and shows where that answer ranks when top-5 succeeds.


In [ ]:
incorrect = probabilities.loc[~probabilities['top_1_correct']].copy()
recovered = incorrect.loc[incorrect['top_5_correct']]
missed = incorrect.loc[~incorrect['top_5_correct']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rank_counts = recovered['answer_rank'].value_counts().sort_index()
axes[0].bar(rank_counts.index.astype(int), rank_counts.values, color='#457b9d')
axes[0].set(
    title='Correct-answer rank among top-1 errors recovered by top-5',
    xlabel='Correct answer rank', ylabel='Examples', xticks=[2, 3, 4, 5],
)

answer_bins = np.linspace(0, max(0.5, incorrect['answer_probability'].quantile(0.995)), 41)
axes[1].hist(
    recovered['answer_probability'].dropna(), bins=answer_bins, density=True,
    histtype='step', linewidth=2, color=colors['Top-5 recovery'], label='Top-5 recovery',
)
axes[1].hist(
    missed['answer_probability'].dropna(), bins=answer_bins, density=True,
    histtype='step', linewidth=2, color=colors['Outside top-5'], label='Outside top-5',
)
axes[1].set(title='Correct-answer probability when top-1 is wrong', xlabel='Correct-answer probability', ylabel='Density')
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'top5_recovery.png', dpi=160, bbox_inches='tight')
plt.show()

rank_counts.rename('examples').to_frame()


## Top-1 calibration

A calibrated model should have observed top-1 accuracy close to its mean top-1 probability in every confidence bin.


In [ ]:
probabilities['confidence_bin'] = pd.cut(
    probabilities['top_1_probability'], bins=np.linspace(0, 1, 11), include_lowest=True
)
calibration = probabilities.groupby('confidence_bin', observed=False).agg(
    examples=('row', 'size'),
    mean_confidence=('top_1_probability', 'mean'),
    observed_accuracy=('top_1_correct', 'mean'),
).dropna().reset_index()

fig, axis = plt.subplots(figsize=(7, 6))
axis.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
axis.plot(calibration['mean_confidence'], calibration['observed_accuracy'], 'o-', color='#264653', label='Model')
for row in calibration.itertuples():
    axis.annotate(f'{row.examples:,}', (row.mean_confidence, row.observed_accuracy), xytext=(4, 4), textcoords='offset points', fontsize=8)
axis.set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean top-1 probability', ylabel='Observed top-1 accuracy', title='Hint-restricted top-1 calibration')
axis.grid(alpha=0.2)
axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'top1_calibration.png', dpi=160, bbox_inches='tight')
plt.show()
calibration


In [ ]:
summary.to_csv(OUTPUT_DIR / 'outcome_probability_summary.csv', index=False)
calibration.to_csv(OUTPUT_DIR / 'top1_calibration.csv', index=False)
rank_counts.rename('examples').to_csv(OUTPUT_DIR / 'top5_recovery_ranks.csv')
print(f'Analysis tables and figures saved under {OUTPUT_DIR}')
